# 1. Tissues-Specific Gene Expression

## 1.1 Reactome (Genes)

In [ ]:
from data.pathways.reactome import ReactomeNetwork

In [ ]:
R = ReactomeNetwork()

In [ ]:
hierarchy = R.reactome.hierarchy
human_hierarchy = hierarchy[hierarchy['child'].str.contains('HSA')]
human_hierarchy.to_csv("../Cytoscape_data/hierarchy.csv")
human_hierarchy

In [ ]:
pathway_genes.groupby('group').

In [ ]:
pathway_genes = R.reactome.pathway_genes
pathway_genes.to_csv("../Cytoscape_data/pathway_genes.csv")
pathway_genes

In [ ]:
pathway_names = R.reactome.pathway_names
human_pathway_names = pathway_names[pathway_names['reactome_id'].str.contains('HSA')]
human_pathway_names.to_csv("../Cytoscape_data/pathway_names.csv")

In [ ]:
import numpy as np
import pandas as pd 

In [ ]:
pathway_genes.columns = ['child', 'parent']

In [ ]:
edges = pd.concat([human_hierarchy,pathway_genes],axis=0)
edges.columns =['TARGET','SOURCE']
edges.to_csv('../Cytoscape_data/edges.csv')

In [ ]:
node = np.concatenate([pathway_genes.gene.unique(), human_pathway_names.reactome_id.unique()])

In [ ]:
edge

In [ ]:
reactome_gene = R.reactome.pathway_genes.gene.unique()

In [ ]:
len(reactome_gene)

In [ ]:
R.ge

## 1.2 FANTOM (Genes & Tisseus)

### 1.2.1 Read Data

In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 

In [ ]:
fantom = pd.read_csv("_database/genes/FantomCAT_robust_genes_vs_traits_pvalue_nonzero_04072021.csv",header=1)

In [ ]:
fantom.head()

In [ ]:
fantom.describe()

In [ ]:
# value_counts of RNAs type
fantom.geneClass.value_counts()
fantom_gene = fantom.geneName.values

In [ ]:
fantom_ex = fantom.iloc[:, 10:] # fantom_ex = FANTOM gene EXpression matrix

fantom_ex_cod = fantom_ex[fantom.geneClass == "coding_mRNA"] # fantom_ex_cod = FANTOM gene EXpression in CODing rna 
fantom_ex_bin = fantom_ex < 0.05 # fantom_ex_bin = FANTOM gene EXpression in BINary if p-value > 0.05 
fantom_ex_log = -np.log(fantom_ex) # fantom_ex_log = FANTOM gene EXpression in LOGarithm 
fantom_ex_sig = fantom_ex[fantom_ex_bin.sum(axis=1) > 0] # fantom_ex_sig = FANTOM gene EXpression which are SIGnificant(p-values < 0.05)

fantom_ex_cod_bin = fantom_ex_cod < 0.05 # fantom_ex_cod_bin = FANTOM gene EXpression in CODing rna in BINary if p-value > 0.05 
fantom_ex_cod_log = -np.log(fantom_ex_cod) # fantom_ex_cod_log = FANTOM gene EXpression in CODing rna in LOGarithm 
fantom_ex_cod_sig = fantom_ex_cod[fantom_ex_cod_bin.sum(axis=1) > 0] # fantom_ex_cod_sig = FANTOM gene EXpression in CODing rna which are SIGnificant(p-values < 0.05)

fantom_ex_sig_bin = fantom_ex_sig < 0.05 # fantom_ex_bin_sig = FANTOM gene EXpression in BINary which are SIGnificant(p-values < 0.05)
fantom_ex_sig_log = -np.log(fantom_ex_sig) # fantom_ex_log_sig = FANTOM gene EXpression in LOGarithm  which are SIGnificant(p-values < 0.05)

fantom_ex_cod_sig_log = -np.log(fantom_ex_cod_sig)
fantom_ex_cod_sig_bin = fantom_ex_cod_sig < 0.05

In [ ]:
# plot of Number of tissues with p-values < 0.05
temp = fantom_ex_bin.sum(axis=1).value_counts().sort_index()
temp.plot(kind="bar",
          xlabel="Number of tissues with p-values < 0.05",
          ylabel="Number of genes")
plt.show()

In [ ]:
# plot of means of p-values in each gene distribution 
fantom_ex.mean(axis=1).hist(bins=50)

In [ ]:
# plot of standard deviation of p-values in each gene distribution 
fantom_ex.std(axis=1).hist(bins=50)

* Note: Means may not be a good choice because you may find some genes that have mean p-values < 1 or pre-defined threshold but are not significant(p-value<0.05)

In [ ]:
# there are 9960 genes with mean of p-values < 1 but are not significant
sum((fantom_ex.mean(axis=1)<1) * (fantom_ex_bin.sum(axis=1) == 0))

### 1.2.2 K-means clustering 

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
num_clusters = 20

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(fantom_ex_bin.T)

In [ ]:
import matplotlib.pyplot as plt
 
# Create a range of values for k
k_range = range(1, 25)
 
# Initialize an empty list to
# store the inertia values for each k
inertia_values = []
 
# Fit and plot the data for each k value
for k in k_range:
    kmeans = KMeans(n_clusters=k, \
                    init='k-means++', random_state=42)
    y_kmeans = kmeans.fit_predict(fantom_ex.T)
    inertia_values.append(kmeans.inertia_)

# Plot the inertia values for each k
plt.plot(k_range, inertia_values, 'bo-')
plt.title('Elbow Method')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.show()

### 1.2.3 PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# standardized_data = StandardScaler().fit_transform(fantom_ex_sig_bin.T)
pca = PCA(n_components=fantom_ex_sig_bin.shape[1]-1)
pca_data = pca.fit_transform(fantom_ex_sig_bin.T)
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:, 0], pca_data[:, 1],c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Result")
plt.show()

In [ ]:
fantom_PCA_df = pd.DataFrame(data=pca_data,index=fantom_ex.columns)

In [ ]:
fantom_PCA_df

### 1.2.4 TSNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, verbose=1, perplexity=20, n_iter=500)
tsne_results = tsne.fit_transform(fantom_ex_bin.T)

In [ ]:
plt.scatter(tsne_results[:, 0], tsne_results[:, 1],c=clusters)
plt.xlabel("TSNE Component 1")
plt.ylabel("TSNE Component 2")
plt.title("TSNE Result")
plt.show()

### 1.2.5 Heatmap

In [ ]:
# Create the heatmap
plt.figure(figsize=(10, 6))
sns.clustermap(fantom_PCA_df.iloc[:50,:10], center=3, fmt=".5f", linewidths=0.5)
plt.title("Gene Expression Heatmap")
plt.xlabel("Samples")
plt.ylabel("Genes")
plt.show()

## 1.3 GTEx

### 1.3.1 Read Data 

In [ ]:
# download data from https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz
gtex = pd.read_csv("_database/genes/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct",sep="\t")

In [ ]:
gtex.head()

In [ ]:
gtex_gene = gtex.Description.values

In [ ]:
gtex.describe()

In [ ]:
# from column index=2 gene expression starts 
gtex.columns[:23]

In [ ]:
gtex_ex = gtex.iloc[:,2:]

### 1.3.2 K-means clustering

In [ ]:
from sklearn.cluster import KMeans
num_clusters = 10

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(gtex_ex.T)

### 1.3.3 PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
standardized_data = StandardScaler().fit_transform(gtex_ex.T)
pca = PCA(n_components=standardized_data.shape[0]-1)
pca_data = pca.fit_transform(standardized_data)
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:, 0], pca_data[:, 1],c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Result")
plt.show()

In [ ]:
fantom_PCA_df = pd.DataFrame(data=pca_data,index=fantom_ex.columns)

In [ ]:
fantom_PCA_df

In [ ]:
gtex_gene = gtex.Description.unique()
reactome_gtex_gene = np.intersect1d(reactome_gene, gtex_gene)
len(reactome_gtex_gene)

In [ ]:
# reactome and gtex intersection genes expression
reactome_gtex_index = [i in reactome_gtex_gene for i in gtex.Description]
reactome_gtex_df = gtex[reactome_gtex_index]

## 1.4 Comparison of Genes

In [ ]:
# import modules
from matplotlib_venn import venn3 
from matplotlib import pyplot as plt

In [ ]:
len(np.intersect1d(reactome_gene,gtex_gene))

In [ ]:
venn = venn3([set(reactome_gene), set(fantom_gene), set(gtex_gene)],
             set_labels=("Reactome", "Fantom","GTEx"))
plt.show()

# 2. Data 

In [ ]:
import numpy as np
import importlib
from data.data_access import Data
from preprocessing import pre
params_file = 'train/params/P1000/pnet/onsplit_average_reg_10_tanh_large_testing.py'

In [ ]:
loader = importlib.machinery.SourceFileLoader('params', params_file)
params = loader.load_module()

In [ ]:
temp = Data(**params.data[0])

In [ ]:
x, y, inf, col = temp.get_data()

In [ ]:
x_train, x_validate_, x_test_, y_train, y_validate_, y_test_, info_train, info_validate_, info_test_, cols = temp.get_train_validate_test()

In [ ]:
import pandas as pd 
tcga = pd.read_csv("_database/genes/tcga_prostate_expressed_genes_and_cancer_genes.csv")
hugo = pd.read_csv("_database/genes/HUGO_genes/protein-coding_gene_with_coordinate_minimal.txt",header=None,sep="\t")
hugo = hugo.iloc[:,3]
tcga = tcga.genes
from data.prostate_paper.data_reader import load_data_type
mut = load_data_type("mut_important", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
mut = mut[0]
print("mut.shape: ",mut.shape)
cnv_amp = load_data_type("cnv_amp", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
cnv_amp = cnv_amp[0]
print("cnv_amp.shape",cnv_amp.shape)
cnv_del = load_data_type("cnv_del", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
cnv_del = cnv_del[0]
print("cnv_del.shape",cnv_del.shape)

mut_cnv = mut.columns.union(cnv_amp.columns)
mut_cnv_tcga = mut_cnv.intersection(tcga)
mut_cnv_tcga_hugo = mut_cnv_tcga.intersection(hugo)
len(mut_cnv_tcga_hugo)

In [ ]:
X = np.concatenate((x_test_, x_validate_, x_train))
Y = np.concatenate((y_test_, y_validate_, y_train))

In [ ]:
x_t = np.concatenate((x_test_, x_validate_))
y_t = np.concatenate((y_test_, y_validate_))
info_t = info_test_.append(info_validate_)

In [ ]:
print("x_train:", x_train.shape)
print("x_train:", y_train.shape)

print("x_validate_:", x_validate_.shape)
print("y_validate_:", y_validate_.shape)

print("x_test:", x_t.shape)
print("y_test:", y_t.shape)

print("columns:", len(cols))

In [ ]:
proc = pre.get_processor(params.pre)
if proc:
    proc.fit(x_train)
    x_train = proc.transform(x_train)
    x_test = proc.transform(x_t)
    if scipy.sparse.issparse(x_train):
        x_train = x_train.todense()
        x_test = x_test.todense()

# 3. Model

In [ ]:
from copy import deepcopy
from model import nn

## 3.1 Trainig 

* Note: Change the `single_outputs` argument in `onsplit_average_reg_10_tanh_large_testing.py` file to True to make single node output
        

In [ ]:
model_params_ = deepcopy(params.models[0])

In [ ]:
model = nn.Model(**model_params_['params'])

In [ ]:
history = model.fit(x_train, y_train)

In [ ]:
test = model.predict(x_t)

In [ ]:
res = y_t == test

In [ ]:
res.sum()/len(y_t)

## 3.2 Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
from statistics import mean, stdev

In [ ]:
model_params_ = deepcopy(params.models[0])

In [ ]:
n_splits = 5  # Number of folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
accurasies = []
for train_indices, val_indices in kf.split(X):
    x_train_fold, x_val_fold = X[train_indices], X[val_indices]
    y_train_fold, y_val_fold = Y[train_indices], Y[val_indices]
    
    model = nn.Model(**model_params_['params'])
    
    model.fit(x_train_fold, y_train_fold)
    test = model.predict(x_val_fold)
    res = y_val_fold == test
    accurasies.append(res.sum()/len(y_val_fold))
    print("Accurasies: ", accurasies)

In [ ]:
accurasies

In [ ]:
print(mean(accurasies)) 
print(stdev(accurasies)) 
# Single Node : 0.8338243183924303 / Multiple Node : 0.8348241720723797
# Single Node : 0.014730575250953129 / Multiple Node : 0.033270867867846915